# Обучение RCAN

## Импорты

In [1]:
import os
import cv2
import math
import random
import numpy as np
from tqdm import tqdm
from pathlib import Path
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from typing import List, Union, Tuple

## Модель


In [2]:
def default_conv(in_channels, out_channels, kernel_size, bias=True):
    return nn.Conv2d(
        in_channels, out_channels, kernel_size,
        padding=(kernel_size//2), bias=bias)

class MeanShift(nn.Conv2d):
    def __init__(self, rgb_range, rgb_mean, rgb_std, sign=-1):
        super(MeanShift, self).__init__(3, 3, kernel_size=1)
        std = torch.Tensor(rgb_std)
        self.weight.data = torch.eye(3).view(3, 3, 1, 1)
        self.weight.data.div_(std.view(3, 1, 1, 1))
        self.bias.data = sign * rgb_range * torch.Tensor(rgb_mean)
        self.bias.data.div_(std)
        self.requires_grad = False

class Upsampler(nn.Sequential):
    def __init__(self, conv, scale, n_feat, bn=False, act=False, bias=True):

        m = []
        if (scale & (scale - 1)) == 0:
            for _ in range(int(math.log(scale, 2))):
                m.append(conv(n_feat, 4 * n_feat, 3, bias))
                m.append(nn.PixelShuffle(2))
                if bn: m.append(nn.BatchNorm2d(n_feat))
                if act: m.append(act())
        elif scale == 3:
            m.append(conv(n_feat, 9 * n_feat, 3, bias))
            m.append(nn.PixelShuffle(3))
            if bn: m.append(nn.BatchNorm2d(n_feat))
            if act: m.append(act())
        else:
            raise NotImplementedError

        super(Upsampler, self).__init__(*m)

@dataclass
class RCANConfig:
    n_resgroups: int = 10
    n_resblocks: int = 20
    n_feats: int = 64
    reduction: int = 16
    scale: Union[int, List[int]] = 2
    res_scale: float = 1.0

    n_colors: int = 3
    rgb_range: int = 255
    rgb_mean: Tuple[float, ...] = (0.4488, 0.4371, 0.4040)
    rgb_std: Tuple[float, ...] = (1.0, 1.0, 1.0)

    kernel_size: int = 3
    use_batch_norm: bool = False

    def __post_init__(self):
        if isinstance(self.scale, int):
            self.scale = [self.scale]


class CALayer(nn.Module):
    def __init__(self, channel: int, reduction: int = 16):
        super(CALayer, self).__init__()
        reduced_channels = max(channel // reduction, 1)

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv_du = nn.Sequential(
            nn.Conv2d(channel, reduced_channels, 1, padding=0, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(reduced_channels, channel, 1, padding=0, bias=True),
            nn.Sigmoid()
        )

    def forward(self, x):
        y = self.avg_pool(x)
        y = self.conv_du(y)
        return x * y


class RCAB(nn.Module):
    def __init__(
        self,
        conv,
        n_feat: int,
        kernel_size: int,
        reduction: int,
        bias: bool = True,
        bn: bool = False,
        act=nn.ReLU(True),
        res_scale: float = 1.0
    ):
        super(RCAB, self).__init__()
        self.res_scale = res_scale

        modules_body = []
        for i in range(2):
            modules_body.append(conv(n_feat, n_feat, kernel_size, bias=bias))
            if bn:
                modules_body.append(nn.BatchNorm2d(n_feat))
            if i == 0:
                modules_body.append(act)
        modules_body.append(CALayer(n_feat, reduction))

        self.body = nn.Sequential(*modules_body)

    def forward(self, x):
        res = self.body(x)
        if self.res_scale != 1.0:
            res = res.mul(self.res_scale)
        return res + x


class ResidualGroup(nn.Module):
    def __init__(
        self,
        conv,
        n_feat: int,
        kernel_size: int,
        reduction: int,
        act,
        res_scale: float,
        n_resblocks: int
    ):
        super(ResidualGroup, self).__init__()

        modules_body = [
            RCAB(
                conv, n_feat, kernel_size, reduction,
                bias=True, bn=False, act=nn.ReLU(True), res_scale=res_scale
            )
            for _ in range(n_resblocks)
        ]
        modules_body.append(conv(n_feat, n_feat, kernel_size))

        self.body = nn.Sequential(*modules_body)

    def forward(self, x):
        res = self.body(x)
        return res + x


class RCAN(nn.Module):
    def __init__(self, args, conv=default_conv):
        super(RCAN, self).__init__()

        if isinstance(args, RCANConfig):
            self.config = args
        else:
            self.config = RCANConfig(
                n_resgroups=getattr(args, 'n_resgroups', 10),
                n_resblocks=getattr(args, 'n_resblocks', 20),
                n_feats=getattr(args, 'n_feats', 64),
                reduction=getattr(args, 'reduction', 16),
                scale=getattr(args, 'scale', 2),
                res_scale=getattr(args, 'res_scale', 1.0),
                n_colors=getattr(args, 'n_colors', 3),
                rgb_range=getattr(args, 'rgb_range', 255),
                kernel_size=getattr(args, 'kernel_size', 3)
            )

        rgb_mean = self.config.rgb_mean
        rgb_std = self.config.rgb_std

        self.sub_mean = MeanShift(self.config.rgb_range, rgb_mean, rgb_std)

        modules_head = [conv(self.config.n_colors, self.config.n_feats, self.config.kernel_size)]
        self.head = nn.Sequential(*modules_head)

        modules_body = [
            ResidualGroup(
                conv,
                self.config.n_feats,
                self.config.kernel_size,
                self.config.reduction,
                act=nn.ReLU(True),
                res_scale=self.config.res_scale,
                n_resblocks=self.config.n_resblocks
            )
            for _ in range(self.config.n_resgroups)
        ]
        modules_body.append(conv(self.config.n_feats, self.config.n_feats, self.config.kernel_size))
        self.body = nn.Sequential(*modules_body)

        scale = self.config.scale[0] if self.config.scale else 1
        modules_tail = [
            Upsampler(conv, scale, self.config.n_feats, act=False),
            conv(self.config.n_feats, self.config.n_colors, self.config.kernel_size)
        ]
        self.tail = nn.Sequential(*modules_tail)

        self.add_mean = MeanShift(self.config.rgb_range, rgb_mean, rgb_std, 1)

    def forward(self, x):
        x = self.sub_mean(x)
        x = self.head(x)

        res = self.body(x)
        res += x

        x = self.tail(res)
        x = self.add_mean(x)

        return x

    def load_state_dict(self, state_dict, strict=False):
        own_state = self.state_dict()
        for name, param in state_dict.items():
            if name in own_state:
                if isinstance(param, nn.Parameter):
                    param = param.data
                try:
                    own_state[name].copy_(param)
                except Exception:
                    if name.find('tail') >= 0:
                        print('Replace pre-trained upsampler to new one...')
                    else:
                        raise RuntimeError('While copying the parameter named {}, '
                                           'whose dimensions in the model are {} and '
                                           'whose dimensions in the checkpoint are {}.'
                                           .format(name, own_state[name].size(), param.size()))
            elif strict:
                if name.find('tail') == -1:
                    raise KeyError('unexpected key "{}" in state_dict'
                                   .format(name))

        if strict:
            missing = set(own_state.keys()) - set(state_dict.keys())
            if len(missing) > 0:
                raise KeyError('missing keys in state_dict: "{}"'.format(missing))

    def count_parameters(self):
        """Подсчет общего количества обучаемых параметров"""
        total_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        non_trainable_params = sum(p.numel() for p in self.parameters() if not p.requires_grad)

        print(f"Total parameters: {total_params:,}")
        print(f"Trainable parameters: {trainable_params:,}")
        print(f"Non-trainable parameters: {non_trainable_params:,}")
        return total_params

## Обучение Модели

### Скачивание датасета DIV2K bicubic downscale

In [ ]:
def download_div2k():
    print("Downloading DIV2K dataset...")

    urls = {
        "train_hr": "https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip",
        "valid_hr": "https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip",
        "train_lr_bicubic_x2": "http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_LR_bicubic_X2.zip",
        "train_lr_bicubic_x3": "http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_LR_bicubic_X3.zip",
        "train_lr_bicubic_x4": "http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_LR_bicubic_X4.zip",
        "valid_lr_bicubic_x2": "http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_LR_bicubic_X2.zip",
        "valid_lr_bicubic_x3": "http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_LR_bicubic_X3.zip",
        "valid_lr_bicubic_x4": "http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_LR_bicubic_X4.zip"
    }

    if not os.path.exists("/content/DIV2K_train_HR"):
        print("Downloading DIV2K_train_HR...")
        !wget -O /content/DIV2K_train_HR.zip {urls['train_hr']}
        print("Extracting DIV2K_train_HR...")
        !unzip -q /content/DIV2K_train_HR.zip -d /content/
        os.remove("/content/DIV2K_train_HR.zip")

    if not os.path.exists("/content/DIV2K_valid_HR"):
        print("Downloading DIV2K_valid_HR...")
        !wget -O /content/DIV2K_valid_HR.zip {urls['valid_hr']}
        print("Extracting DIV2K_valid_HR...")
        !unzip -q /content/DIV2K_valid_HR.zip -d /content/
        os.remove("/content/DIV2K_valid_HR.zip")

    lr_dirs = {
        "train": ["train_lr_bicubic_x2", "train_lr_bicubic_x3", "train_lr_bicubic_x4"],
        "valid": ["valid_lr_bicubic_x2", "valid_lr_bicubic_x3", "valid_lr_bicubic_x4"]
    }

    for lr_type in lr_dirs['train']:
        dest_dir = f"/content/DIV2K_{lr_type.upper()}"
        if not os.path.exists(dest_dir):
            print(f"Downloading {lr_type}...")
            !wget -O /content/{lr_type}.zip {urls[lr_type]}
            print(f"Extracting {lr_type}...")
            !unzip -q /content/{lr_type}.zip -d /content/
            os.remove(f"/content/{lr_type}.zip")

    for lr_type in lr_dirs['valid']:
        dest_dir = f"/content/DIV2K_{lr_type.upper()}"
        if not os.path.exists(dest_dir):
            print(f"Downloading {lr_type}...")
            !wget -O /content/{lr_type}.zip {urls[lr_type]}
            print(f"Extracting {lr_type}...")
            !unzip -q /content/{lr_type}.zip -d /content/
            os.remove(f"/content/{lr_type}.zip")

    print("Dataset download completed!")

download_div2k()

--2026-04-25 18:20:27--  https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip
Resolving data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)... 129.132.52.178, 2001:67c:10ec:36c2::178
Connecting to data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)|129.132.52.178|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3530603713 (3.3G) [application/zip]
Saving to: ‘/content/DIV2K_train_HR.zip’

t/DIV2K_train_HR.zi   2%[                    ]  76.51M  22.3MB/s    eta 3m 21s ^C
Extracting DIV2K_train_HR...
[/content/DIV2K_train_HR.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/DIV2K_train_HR.zip or
        /content/DIV2K_train_HR.zip.zip, and cannot find /content/DIV2K_train_HR.zip.ZIP, period.
--2026-04-25

### Код обучения

#### Класс датасета

In [3]:
class DIV2KDataset(Dataset):
    def __init__(self, hr_dir, lr_dir, scale, patch_size=96, is_train=True, samples_per_epoch=800):
        self.scale = scale
        self.patch_size_hr = patch_size
        self.patch_size_lr = patch_size // scale
        self.is_train = is_train
        self.samples_per_epoch = samples_per_epoch if is_train else 112

        self.hr_paths = sorted([str(p) for p in Path(hr_dir).glob("*.png")])
        if not self.hr_paths:
            self.hr_paths = sorted([str(p) for p in Path(hr_dir).glob("*.jpg")])

        lr_subdir = Path(lr_dir) / f"X{scale}"
        self.lr_paths = []
        for hr_path in self.hr_paths:
            img_name = Path(hr_path).stem
            lr_path = lr_subdir / f"{img_name}x{scale}.png"
            if not lr_path.exists():
                lr_path = lr_subdir / f"{img_name}.png"
            self.lr_paths.append(str(lr_path) if lr_path.exists() else None)

        print(f"Loaded {len(self.hr_paths)} images")

    def __len__(self):
        return self.samples_per_epoch

    def __getitem__(self, idx):
        img_idx = random.randint(0, len(self.hr_paths) - 1)

        img_hr = cv2.imread(self.hr_paths[img_idx], cv2.IMREAD_COLOR)
        img_hr = cv2.cvtColor(img_hr, cv2.COLOR_BGR2RGB)

        lr_path = self.lr_paths[img_idx]
        if lr_path and Path(lr_path).exists():
            img_lr = cv2.imread(lr_path, cv2.IMREAD_COLOR)
            img_lr = cv2.cvtColor(img_lr, cv2.COLOR_BGR2RGB)
        else:
            h, w = img_hr.shape[:2]
            img_lr = cv2.resize(img_hr, (w//self.scale, h//self.scale), interpolation=cv2.INTER_CUBIC)

        h, w = img_hr.shape[:2]
        if h >= self.patch_size_hr and w >= self.patch_size_hr:
            ph = random.randint(0, h - self.patch_size_hr)
            pw = random.randint(0, w - self.patch_size_hr)
            img_hr = img_hr[ph:ph+self.patch_size_hr, pw:pw+self.patch_size_hr]
            img_lr = img_lr[ph//self.scale:(ph+self.patch_size_hr)//self.scale,
                           pw//self.scale:(pw+self.patch_size_hr)//self.scale]

        if self.is_train and random.random() > 0.5:
            img_hr = np.fliplr(img_hr).copy()
            img_lr = np.fliplr(img_lr).copy()

        img_hr = img_hr.astype(np.float32)
        img_lr = img_lr.astype(np.float32)

        img_hr = torch.from_numpy(img_hr).permute(2, 0, 1).float()
        img_lr = torch.from_numpy(img_lr).permute(2, 0, 1).float()

        return img_lr, img_hr

#### Метрики

In [4]:
def calculate_psnr(img1, img2):
    mse = torch.mean((img1 - img2) ** 2)
    if mse == 0:
        return 100.0
    psnr = 20 * torch.log10(255.0 / torch.sqrt(mse))
    return psnr.item()

def gaussian(window_size, sigma):
    gauss = torch.Tensor([np.exp(-(x - window_size//2)**2/float(2*sigma**2)) for x in range(window_size)])
    return gauss/gauss.sum()

def create_window(window_size, channel):
    _1D_window = gaussian(window_size, 1.5).unsqueeze(1)
    _2D_window = _1D_window.mm(_1D_window.t()).float().unsqueeze(0).unsqueeze(0)
    return _2D_window.expand(channel, 1, window_size, window_size).contiguous()

def calculate_ssim(img1, img2, window_size=11):
    channel = img1.size(1)
    window = create_window(window_size, channel).to(img1.device)

    mu1 = nn.functional.conv2d(img1, window, padding=window_size//2, groups=channel)
    mu2 = nn.functional.conv2d(img2, window, padding=window_size//2, groups=channel)

    mu1_sq, mu2_sq = mu1.pow(2), mu2.pow(2)
    mu1_mu2 = mu1 * mu2

    sigma1_sq = nn.functional.conv2d(img1*img1, window, padding=window_size//2, groups=channel) - mu1_sq
    sigma2_sq = nn.functional.conv2d(img2*img2, window, padding=window_size//2, groups=channel) - mu2_sq
    sigma12 = nn.functional.conv2d(img1*img2, window, padding=window_size//2, groups=channel) - mu1_mu2

    C1 = (0.01 * 255) ** 2
    C2 = (0.03 * 255) ** 2

    ssim_map = ((2*mu1_mu2 + C1)*(2*sigma12 + C2)) / ((mu1_sq+mu2_sq+C1)*(sigma1_sq+sigma2_sq+C2))
    return ssim_map.mean().item()

#### Функция тренировки

In [6]:
def train_epoch(epoch,
                scale,
                model,
                train_loader,
                val_loader,
                optimizer,
                criterion,
                scheduler,
                device):
    print(f"\nEpoch {epoch+1}/200")

    model.train()
    train_loss = 0
    pbar = tqdm(train_loader, desc="Training")
    for lr, hr in pbar:
        lr, hr = lr.to(device), hr.to(device)
        optimizer.zero_grad()
        sr = model(lr)
        loss = criterion(sr, hr)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    model.eval()
    val_loss, val_psnr, val_ssim = 0, 0, 0
    pbar = tqdm(val_loader, desc="Validation")
    with torch.no_grad():
        for lr, hr in pbar:
            lr, hr = lr.to(device), hr.to(device)
            sr = model(lr)
            loss = criterion(sr, hr)
            psnr_val = calculate_psnr(sr, hr)
            ssim_val = calculate_ssim(sr, hr)

            val_loss += loss.item()
            val_psnr += psnr_val
            val_ssim += ssim_val
            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'psnr': f'{psnr_val:.2f}dB',
                'ssim': f'{ssim_val:.4f}'
            })

    val_loss /= len(val_loader)
    val_psnr /= len(val_loader)
    val_ssim /= len(val_loader)
    scheduler.step()

    print(f"Epoch {epoch+1}: Val Loss={val_loss:.4f}, PSNR={val_psnr:.2f}dB, SSIM={val_ssim:.4f}")

    if val_psnr > best_psnr:
        best_psnr = val_psnr
        torch.save(model.state_dict(), f"/content/checkpoints/X{scale}/rcan_x{scale}_best.pth")
        print(f"New best model. PSNR={best_psnr:.2f}dB")

    if (epoch+1) % 10 == 0:
        torch.save(model.state_dict(), f"/content/checkpoints/X{scale}/rcan_x{scale}_epoch{epoch+1}.pth")


def train(scale: int = 2, epoch: int = 100):
    print(f"\n{'='*60}")
    print(f"Training RCAN for x{scale}")
    print(f"{'='*60}\n")
    patch_size = scale * 48

    config = RCANConfig(scale=scale)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")

    train_dataset = DIV2KDataset(
        "/content/DIV2K_train_HR", "/content/DIV2K_train_LR_bicubic",
        scale, patch_size, is_train=True, samples_per_epoch=800
    )
    val_dataset = DIV2KDataset(
        "/content/DIV2K_valid_HR", "/content/DIV2K_valid_LR_bicubic",
        scale, patch_size, is_train=False, samples_per_epoch=112
    )

    train_loader = DataLoader(train_dataset, batch_size=16, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=16, num_workers=4, pin_memory=True)

    model = RCAN(config).to(device)
    print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

    criterion = nn.L1Loss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.5)

    best_psnr = 0

    os.makedirs("/content/checkpoints", exist_ok=True)
    os.makedirs(f"/content/checkpoints/X{scale}", exist_ok=True)

    for epoch in range(200):
        train_epoch(
            epoch, scale, model,
            train_loader, val_loader,
            optimizer, criterion, criterion,
            device
        )

    print(f"\n{'='*60}")
    print(f"Training completed! Best PSNR: {best_psnr:.2f}dB")
    print(f"{'='*60}")

### Обучение

#### Масштабирование Х2

In [ ]:
train(scale=2, epoch=100)

#### Масштабирование Х3

In [ ]:
train(scale=3, epoch=150)

#### Масштабирование Х4

In [ ]:
train(scale=4, epoch=200)